# Soccer Analytics — STVN Training
Run on **GPU runtime** (Runtime → Change runtime type → T4 GPU)

In [ ]:
# 1. Clone repo
!git clone https://github.com/vishsriram10/leverkusen_analysis_project.git
%cd leverkusen_analysis_project

In [ ]:
# 2. Install dependencies (PyG needs special install on Colab)
!pip install -q statsbombpy socceraction xgboost joblib matplotlib
!pip install -q torch-geometric
print('Done')

In [ ]:
# 3. Verify GPU
import torch
print('GPU available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device:', torch.cuda.get_device_name(0))

In [ ]:
# 4. Ingest 360 data (~10-15 min)
!python stvn/ingest.py

In [ ]:
# 5. Build possession chains (~5 min)
!python stvn/build_chains.py

In [ ]:
# 6. (Optional) Save chains to Drive so you don't rebuild next session
from google.colab import drive
drive.mount('/content/drive')
!cp data/360/chains.pkl /content/drive/MyDrive/soccer_analytics_chains.pkl
print('Chains saved to Drive')

In [ ]:
# 6b. (If resuming) Load chains from Drive instead of rebuilding
# from google.colab import drive
# drive.mount('/content/drive')
# import os; os.makedirs('data/360', exist_ok=True)
# !cp /content/drive/MyDrive/soccer_analytics_chains.pkl data/360/chains.pkl

In [ ]:
# 7. Train STVN (~20-40 min on T4)
!python stvn/train.py

In [ ]:
# 8. Save trained model to Drive
!cp models/stvn.pt /content/drive/MyDrive/stvn.pt
print('Model saved to Drive')

In [2]:
# Inspect repository structure and identify key scripts/directories
import os
from pathlib import Path

cwd = Path.cwd()
print('Current working directory:', cwd)

# Show top-level files/directories
items = sorted(cwd.iterdir(), key=lambda p: (p.is_file(), p.name.lower()))
print('\nTop-level contents:')
for p in items:
    kind = 'DIR ' if p.is_dir() else 'FILE'
    print(f'  {kind} {p.name}')

# If repo is nested (e.g., leverkusen_analysis_project), show its structure
candidate_dirs = [p for p in items if p.is_dir() and (p / '.git').exists() or (p / 'pyproject.toml').exists() or (p / 'setup.py').exists()]
if candidate_dirs:
    print('\nCandidate project roots:')
    for p in candidate_dirs:
        print('  ', p)

# Also look for the expected scripts in any subdir
expected = ['ingest_360.py', 'build_chains.py', 'train_stvn.py']
found = []
for root, dirs, files in os.walk(cwd):
    for fn in expected:
        if fn in files:
            found.append(Path(root) / fn)

print('\nExpected pipeline scripts found:')
for p in sorted(set(found)):
    print('  ', p.relative_to(cwd))

# Brief tree (depth=2) for likely project root if present
proj = cwd / 'leverkusen_analysis_project'
if proj.exists() and proj.is_dir():
    print('\nTree for leverkusen_analysis_project (depth=2):')
    for p in sorted(proj.rglob('*')):
        rel = p.relative_to(proj)
        if len(rel.parts) <= 2:
            print('  ', rel, '/' if p.is_dir() else '')
else:
    print("\nNote: 'leverkusen_analysis_project' directory not present here (cells may not have been executed in this environment).")


Current working directory: /Users/vishsriram/projects/soccer-analytics/notebooks

Top-level contents:
  FILE colab_train.ipynb

Expected pipeline scripts found:

Note: 'leverkusen_analysis_project' directory not present here (cells may not have been executed in this environment).


In [3]:
# Clone the project repo into this environment (idempotent: won't reclone if folder exists)
from pathlib import Path
import os

repo_url = "https://github.com/vishsriram10/leverkusen_analysis_project.git"
repo_dir = Path("leverkusen_analysis_project")

if not repo_dir.exists():
    !git clone {repo_url}
else:
    print(f"Repo already exists at: {repo_dir.resolve()}")

# Show top-level contents after clone
print("\nRepo top-level contents:")
for p in sorted(repo_dir.iterdir(), key=lambda p: (p.is_file(), p.name.lower())):
    print("  ", ("DIR " if p.is_dir() else "FILE"), p.name)


Cloning into 'leverkusen_analysis_project'...
Username for 'https://github.com': 

OSError: [Errno 5] Input/output error